# 融合中的同步与缓冲管理

融合 Kernel 的同步与缓冲错误可能导致数据竞争或死锁。本节把易错点整理成规则清单。


## 1. 事件规则清单

1. **成对**：每个 set 必有配对的 wait（同 event_id、同 sync_mode）；
2. **动态配对**：参与端在相应迭代中执行匹配的 set/wait；可以有协调一致的条件分支，不能只让一端跳过；
3. **不复用未消费的 ID**：同一 ID 再次用于另一组依赖前，前一组 SET/WAIT 必须全部完成；
4. **pipe 要覆盖需要同步的数据访问**：Cube 写 GM 后由 FIX 发，Vector 写 GM 后由 MTE3 发；读 GM 前由 MTE2 等。L0C→UB 后接 Vector 计算时可用 FIX→V；若把等待放在更早的流水，还须由核内依赖保证后续访问不会越过等待；
5. **sync_mode 按参与者选**：AIC↔本组两个 AIV 用 INTRA_BLOCK；只对一个 AIV 用 UNICAST_BLOCK；两个 AIV 之间用 INTER_SUBBLOCK；多个逻辑 Block 的同类核全核同步用 INTER_BLOCK，该模式不在 AIC 与 AIV 之间同步。


## 2. 双向同步与多槽缓冲

**循环复用同一缓冲必须双向同步**（第 2 章结论在融合中的强化）：

- 正向 READY：消费者在数据写完后读；
- 反向 RELEASE：生产者在消费者读完后覆盖。

在双向同步基础上，可增加**多槽缓冲**——用 `task_id % depth` 选槽位，相邻任务天然错开：

```text
slot_idx = task_idx % depth
tile = shared_group[slot_idx]
READY = READY_IDS[slot_idx]
RELEASE = RELEASE_IDS[slot_idx]
```

槽位映射让"哪个任务用哪块缓冲、等哪个事件"变成可推导的设计，而不是靠运行时碰运气。


## 3. 上下文的循环缓冲

流水中，延迟阶段（如 Vector 滞后 Cube 一个任务）常需要**前面任务的坐标、偏移、valid_shape**——这些标量上下文用 `pl.struct_array` 保存并按 task_id 轮转，与数据槽位同步轮转。简单两阶段流水能从 task_id 直接算出地址时，可以不用单独的上下文缓冲。


## 4. 自动 CV 并行流水（预告）

手动编排流水是理解融合的正确路径；框架同时提供**自动 CV 并行流水**（`@pl.pipeline.stage` + 跨核缓冲的 fwd_ids/bwd_ids，并在 `@pl.jit` 中设置 `pipeline=pl.pipeline.PipelineConfig(preload=2)` 等配置），在满足使用约束时由框架完成流水排布与核间同步。第 8 章性能优化将展开——**先会手动，再用自动**，出问题时才有能力定位。


## 5. 事件规划的完整算例

场景：Cube 逐 Tile 产 4 块中间结果到 workspace，Vector 逐 Tile 消费（第 2 章 02.06 的同步练习 的落地版）。**三槽缓冲 + 双向事件**的完整规划：

```text
槽位：  slot = task_id % 3        （task_id = Cube 的产出序号 0,1,2,3）
缓冲：  workspace 按槽位分 3 段   （互不重叠）

事件表（event_id ∈ [0,16)）：
  READY[0..2]  = {0, 1, 2}     正向：Cube 写完 slot s → set READY[s]
  RELEASE[0..2]= {3, 4, 5}     反向：Vector 读完 slot s → set RELEASE[s]

时序（tick 为流水轮次）：
tick 0:  Cube 写 slot0 → set READY[0]
tick 1:  Cube 写 slot1 → set READY[1]；  Vector wait READY[0] 读 slot0 → set RELEASE[0]
tick 2:  Cube 写 slot2 → set READY[2]；Vector 消费 slot1 → set RELEASE[1]
tick 3:  Cube wait RELEASE[0] 后复用 slot0 → set READY[0]；Vector 消费 slot2 → set RELEASE[2]
tick 4:  Cube 无新任务；Vector 消费 slot0 → set RELEASE[0]
收尾：消费尚未等待的 RELEASE[1]、RELEASE[2] 以及最后一次 RELEASE[0]。
首次使用槽位不等待尚未产生的 RELEASE。
```

**验证规划正确性的三个问题**（每个多槽设计都自问）：

1. 每个槽位的覆写是否都在 RELEASE 之后？（反向同步覆盖所有复用点）
2. 每个 wait 是否有对应参与者在同次动态迭代发出 set？（避免死锁）
3. event_id 是否有"未消费就复用"？（6 个 ID 恰好够 3 槽双向——加槽就要加 ID）

**ID 预算**：本例 3 槽用 6 个 ID。还须避开其他手动同步及自动流水保留的 ID；超预算时优先减少槽位。复用 ID 必须证明事件生命周期不重叠，不能仅凭 task_id 不同就认为安全。


## 6. 排错工具预告

同步类 bug（死锁、数据竞争）在 950 上的排查利器是 **sanitizer**（第 8 章）：检测 GM 越界、Tile 越界、Tile 区间重叠与 mutex 未正确配对。


### 两槽的四个任务：把首尾也列出来

取 READY={0,1}、RELEASE={2,3}，slot=task%2。任务 0/1 首次使用槽位，Cube 不等待 RELEASE；任务 2 覆写槽 0 前等事件 2，任务 3 覆写槽 1 前等事件 3。每个 AIV 在同一任务的 READY 到达后读取自己负责的 64 行，后处理并写回后发送 RELEASE。INTRA_BLOCK 会等待两个 AIV 的配对信号。

这里有意在 Vector 的 MTE3 写回后释放，便于解释单槽串行基线；可以进一步研究提前到 workspace 读完后释放，但需重新证明 UB 复用与计算依赖。最后两个 RELEASE 仍须被 Cube 消费，不能在循环结束时留下未配对事件。

4 个任务对每槽产生两次 READY 与两次 RELEASE；扩大任务数时，各槽的生产、消费、复用次数仍应一致。slot 只选择存储位置，不构成同步。


## 7. 课后练习

1. （判断）生产端总是 set，配对消费端却可能因 if 分支跳过 wait，这种设计是安全的。（对/错）
2. （填空）多槽缓冲中，槽位选择公式是 slot = ______；它同时决定了 ______ 和 ______ 的编号。
3. （实践）四个独立任务各经过 C1→V→C3，每阶段耗时 1 拍，C1/C3 共用同一个 Cube、V 使用独立 Vector。说明为什么不能套用 n+2 拍公式，并画出满足资源与数据依赖的调度表。真实 SDPA 还要额外考虑跨 KV 块的在线状态依赖。

In [ ]:
!cat ./answer/05.03_answer.txt

## 8. 本节小结

- 事件五规则：成对、动态迭代匹配、不复用未消费 ID、pipe 对齐、mode 按参与者；
- 多槽缓冲仍须双向同步（`task_id % depth` 可推导设计）；
- 上下文用 struct_array 随槽位轮转；
- 自动 CV 流水第 8 章见——先手动后自动。